# 🔥 Algerian Forest Fire Prediction using Machine Learning

**Predicting wildfire occurrence in Algeria from meteorological data and Fire Weather Index (FWI) components.**

This notebook is fully self-contained — click **Run all** (`Runtime > Run all`) and it will download the data, clean it, train two models, and reproduce all results and figures from the accompanying paper/GitHub repo.

- Dataset: [Algerian Forest Fires Dataset](https://archive.ics.uci.edu/dataset/547/algerian+forest+fires+dataset) (UCI ML Repository, Abid & Izeboudjen, 2019)
- Models: Random Forest vs. Logistic Regression baseline
- Result: **98% accuracy**, ROC-AUC = 1.000 (see Limitations section for an honest discussion of why)


## 1. Setup

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, io, zipfile

sns.set_theme(style="whitegrid")
print("Libraries ready.")


## 2. Load the Dataset

We download the official dataset directly from the UCI Machine Learning Repository.
If the automatic download fails (e.g. no internet access in this environment), a file-upload widget will appear — just upload `Algerian_forest_fires_dataset_UPDATE.csv` manually.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/547/algerian+forest+fires+dataset.zip"
RAW_PATH = "Algerian_forest_fires_dataset_UPDATE.csv"

def download_dataset():
    try:
        r = requests.get(DATA_URL, timeout=15)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(csv_name) as f, open(RAW_PATH, "wb") as out:
            out.write(f.read())
        print(f"Downloaded and extracted: {RAW_PATH}")
        return True
    except Exception as e:
        print(f"Automatic download failed ({e}).")
        return False

if not download_dataset():
    try:
        from google.colab import files
        print("Please upload Algerian_forest_fires_dataset_UPDATE.csv:")
        uploaded = files.upload()
        RAW_PATH = list(uploaded.keys())[0]
    except ImportError:
        print("Not running in Colab — place the CSV in the working directory manually and set RAW_PATH.")


## 3. Clean & Combine the Data

The raw CSV has an unusual structure: two regions (Bejaia, Sidi Bel-Abbes) are stacked
in the same file, each with its own header row. There is also a known formatting glitch
in the original file — one row is missing a comma between two numeric fields. The parser
below detects and repairs this automatically (no manual editing needed).

In [ ]:
def fix_row(row, n_cols):
    """Repairs rows affected by a missing comma in the original file
    (e.g. '14.6 9' should be two separate fields '14.6' and '9').
    This makes the row too SHORT by one field, so we find the field
    containing a stray space and split it in two."""
    row = list(row)
    while len(row) < n_cols:
        fixed = False
        for i, field in enumerate(row):
            parts = field.strip().split()
            if len(parts) == 2:
                row = row[:i] + parts + row[i+1:]
                fixed = True
                break
        if not fixed:
            break
    return row


def load_and_clean(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    header_idx, split_idx = None, None
    for i, line in enumerate(lines):
        if "day" in line.lower() and "month" in line.lower():
            if header_idx is None:
                header_idx = i
            else:
                split_idx = i

    columns = [c.strip() for c in lines[header_idx].split(",")]
    n_cols = len(columns)

    def parse_block(block_lines, region_name):
        rows = []
        for l in block_lines:
            l = l.strip()
            if not l or "," not in l:
                continue
            fields = fix_row(l.split(","), n_cols)
            if len(fields) == n_cols:
                rows.append(fields)
        df = pd.DataFrame(rows, columns=columns)
        df["Region"] = region_name
        return df

    block1 = lines[header_idx + 1: split_idx - 1] if split_idx else lines[header_idx + 1:]
    df1 = parse_block(block1, "Bejaia")

    df2 = pd.DataFrame()
    if split_idx:
        block2 = lines[split_idx + 1:]
        df2 = parse_block(block2, "Sidi Bel-Abbes")

    df = pd.concat([df1, df2], ignore_index=True)
    df.columns = [c.strip() for c in df.columns]

    numeric_cols = ["day", "month", "year", "Temperature", "RH", "Ws",
                     "Rain", "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Classes"] = df["Classes"].str.strip().str.lower()
    df["fire_binary"] = df["Classes"].apply(lambda x: 1 if "not" not in str(x) else 0)
    df = df.dropna(subset=["Temperature"])
    return df


df = load_and_clean(RAW_PATH)
print(f"Shape: {df.shape}")
print(df["Classes"].value_counts())
df.head()


## 4. Quick Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["Classes"].value_counts().plot(kind="bar", ax=axes[0], color=["#d62728", "#2ca02c"])
axes[0].set_title("Class Distribution")
axes[0].set_xticklabels(["Fire", "Not Fire"], rotation=0)

sns.boxplot(data=df, x="Classes", y="Temperature", ax=axes[1])
axes[1].set_title("Temperature by Class")

plt.tight_layout()
plt.show()


## 5. Train Models

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score, roc_curve)

FEATURES = ["Temperature", "RH", "Ws", "Rain", "FFMC", "DMC", "DC", "ISI", "BUI"]
TARGET = "fire_binary"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
y_proba_lr = lr.predict_proba(X_test_s)[:, 1]

cv_scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

def report(name, y_true, y_pred, y_proba):
    print(f"{name}")
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"  Recall   : {recall_score(y_true, y_pred):.3f}")
    print(f"  F1-Score : {f1_score(y_true, y_pred):.3f}")
    print(f"  ROC-AUC  : {roc_auc_score(y_true, y_proba):.3f}\n")

report("Random Forest", y_test, y_pred_rf, y_proba_rf)
report("Logistic Regression", y_test, y_pred_lr, y_proba_lr)
print(f"5-fold CV Accuracy (Random Forest): {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")


## 6. Results — Figures

In [ ]:
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index,
            palette="YlOrRd_r", legend=False)
plt.title("Feature Importance for Wildfire Prediction (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
print(importance)


In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Fire", "Fire"], yticklabels=["Not Fire", "Fire"])
plt.title("Confusion Matrix - Random Forest")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_rf)
auc = roc_auc_score(y_test, y_proba_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})", color="darkorange", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_rf, target_names=["Not Fire", "Fire"]))


## 7. Limitations (read before citing these results)

- **Small dataset** (244 records, one fire season, two regions) — results need validation on larger/more recent data.
- **ISI and FFMC are themselves derived from temperature/humidity/wind** via the Canadian FWI system, which partly explains the very high separability (AUC ≈ 1.000). This reflects the structure of the target definition, not purely "free" predictive signal.
- Generalization to other Algerian wilayas or other years is untested.

## Citation

Abid, F., Izeboudjen, N. (2019). *Predicting Forest Fire in Algeria Using Data Mining Techniques: Case Study of the Decision Tree Algorithm*. UCI Machine Learning Repository. https://doi.org/10.24432/C5KW4N
